# 06 -- Latent Trade Price Wine Price Indices (MCMC/Kalman)

**Purpose**: Build a custom price index from WineFi latent trade price model data to
address the methodological critique that Liv-ex indices are liquidity-selected and
survivorship-biased. The latent trade price model (MCMC/Kalman) produces smoother,
MCMC-modelled price estimates at LWIN11 level, aggregated to LWIN7 for index construction.

**Data source**: `dev_winefi_raf.ml.ml_latent_prices_historic` (MotherDuck)

**Linear**: WIN-38 https://linear.app/winefi/issue/WIN-38

## Sections
1. Environment setup
2. Load data (latent prices from MotherDuck + Liv-ex parquets from notebook 01)
3. Dynamic column detection
4. S&P 500 (GBP) via FX conversion
5. Build latent price index
6. Methodology documentation
7. Chart 1 -- Top-10 constituent series
8. Chart 2 -- Latent price index vs Liv-ex (rebased Jan 2005)
9. Chart 3 -- Rolling 12m return spread
10. Chart 4 -- Crisis deep-dive (3 windows)
11. Chart 5 -- Crisis bar comparison
12. Data quality assertions

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Ensure the repo root is in sys.path so project modules can be imported
# regardless of whether this notebook is opened from the repo root or the
# notebook directory itself.
# ---------------------------------------------------------------------------
def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'pyproject.toml').exists() or (parent / '.git').exists():
            return parent
    raise RuntimeError(
        'Could not find repo root (no pyproject.toml or .git found). '
        f'Searched from: {start}'
    )

_repo_root = str(_find_repo_root(Path.cwd()))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f'Repo root: {_repo_root}')

In [ ]:
import os
import warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for CI/headless environments
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import yfinance as yf
from pathlib import Path

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Paths -- notebook lives in projects/correlation-diversification/notebooks/
# When Jupyter runs, CWD is the notebook directory.
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path('__file__').resolve().parent   # .../notebooks/
PROJECT_DIR  = NOTEBOOK_DIR.parent                  # .../correlation-diversification/
DATA_DIR     = PROJECT_DIR / 'data'
IMAGES_DIR   = PROJECT_DIR / 'images' / 'custom_indices'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_PARQUET      = DATA_DIR / 'livex_indices_clean.parquet'
COMPARISON_PARQUET = DATA_DIR / 'comparison_assets_monthly.parquet'

# ---------------------------------------------------------------------------
# WineFi brand colours
# ---------------------------------------------------------------------------
WINEFI_COLOURS = [
    '#9437ff',  # purple  -- primary
    '#83D483',  # mantis
    '#FFD166',  # sunglow
    '#F78C6B',  # coral
    '#4D87D0',  # blue
    '#EF476F',  # red
    '#06D6A0',  # emerald
    '#C23FB7',  # pink/purple
    '#4A4A68',  # slate
]

COLOURS = {
    'purple':  WINEFI_COLOURS[0],
    'green':   WINEFI_COLOURS[1],
    'yellow':  WINEFI_COLOURS[2],
    'orange':  WINEFI_COLOURS[3],
    'blue':    WINEFI_COLOURS[4],
    'red':     WINEFI_COLOURS[5],
    'teal':    WINEFI_COLOURS[6],
    'magenta': WINEFI_COLOURS[7],
    'slate':   WINEFI_COLOURS[8],
}

# Crisis / stress periods (3 windows for charts 4 & 5)
STRESS_PERIODS = [
    {'label': 'GFC 2008',        'start': '2007-10-01', 'end': '2009-06-30', 'colour': COLOURS['red']},
    {'label': 'COVID 2020',      'start': '2020-02-01', 'end': '2020-09-30', 'colour': COLOURS['orange']},
    {'label': '2022 Rate Rises', 'start': '2022-01-01', 'end': '2022-12-31', 'colour': COLOURS['blue']},
]

MIN_CONTRIBUTORS = 10  # minimum LWIN7s required to anchor rebase date
REBASE_DATE      = '2005-01-01'  # for chart 2 comparative rebasing

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})

print('Project dir:  ', PROJECT_DIR)
print('Images dir:   ', IMAGES_DIR)
print('livex parquet exists:      ', LIVEX_PARQUET.exists())
print('comparison parquet exists: ', COMPARISON_PARQUET.exists())

## 2. Load Data

**Latent prices** come from MotherDuck (`dev_winefi_raf.ml.ml_latent_prices_historic`).
Query: `SELECT lwin11, yyyymm, price_latent, region FROM dev_winefi_raf.ml.ml_latent_prices_historic`

LWIN11 -> LWIN7 aggregation: median `price_latent` across vintages per (LWIN7, yyyymm).
`price_latent` is MCMC-modelled -- treated as the price index directly, no further smoothing.

**Benchmark data** (Liv-ex indices, S&P 500, FTSE 100) still loaded from parquets produced
by `01_data_setup.ipynb`.

In [ ]:
# ---------------------------------------------------------------------------
# Connect to MotherDuck and query latent price data
# ---------------------------------------------------------------------------
con = duckdb.connect('md:')
print('Connected to MotherDuck')

latent_raw = con.execute(
    'SELECT lwin11, yyyymm, price_latent, region'
    ' FROM dev_winefi_raf.ml.ml_latent_prices_historic'
).df()

print(f'Latent prices raw: {latent_raw.shape}')
print(f'  Columns: {list(latent_raw.columns)}')
print(f'  LWIN11 count: {latent_raw["lwin11"].nunique()}')
print(f'  yyyymm range: {latent_raw["yyyymm"].min()} -> {latent_raw["yyyymm"].max()}')

# Derive LWIN7 from LWIN11 (first 7 characters)
latent_raw['lwin7'] = latent_raw['lwin11'].astype(str).str[:7]

# Convert yyyymm integer (e.g. 202301) -> month-end date
latent_raw['month'] = (
    pd.to_datetime(latent_raw['yyyymm'].astype(str), format='%Y%m')
    + pd.offsets.MonthEnd(0)
)

# ---------------------------------------------------------------------------
# Aggregate LWIN11 -> LWIN7: median price_latent across vintages per (lwin7, month)
# ---------------------------------------------------------------------------
lwin7_monthly = (
    latent_raw
    .groupby(['lwin7', 'month'])['price_latent']
    .median()
    .reset_index()
    .rename(columns={'price_latent': 'latent_price'})
)

print(f'\nLWIN7 monthly latent prices: {lwin7_monthly.shape}')
print(f'  LWIN7 count: {lwin7_monthly["lwin7"].nunique()}')
print(f'  Date range:  {lwin7_monthly["month"].min().date()} -> {lwin7_monthly["month"].max().date()}')
print()

# ---------------------------------------------------------------------------
# Benchmark parquets from 01_data_setup.ipynb
# ---------------------------------------------------------------------------
for path in [LIVEX_PARQUET, COMPARISON_PARQUET]:
    if not path.exists():
        raise FileNotFoundError(
            f'Required parquet not found: {path}\n'
            'Run projects/correlation-diversification/notebooks/01_data_setup.ipynb first.'
        )

livex_raw = pd.read_parquet(LIVEX_PARQUET)
livex_raw.index = pd.to_datetime(livex_raw.index)
livex_raw = livex_raw[livex_raw.index >= '2000-01-01']
print(f'livex_indices_clean: {livex_raw.shape}')
print(f'  Columns: {list(livex_raw.columns)}')
print()

comp_raw = pd.read_parquet(COMPARISON_PARQUET)
comp_raw.index = pd.to_datetime(comp_raw.index)
comp_raw = comp_raw[comp_raw.index >= '2000-01-01']
print(f'comparison_assets_monthly: {comp_raw.shape}')
print(f'  Columns: {list(comp_raw.columns)}')

## 3. Dynamic Column Detection

Resolve Liv-ex 100, Liv-ex 1000, S&P 500, and FTSE 100 column names dynamically
to guard against naming changes in the upstream CSV or parquet.

In [3]:
numeric_livex = livex_raw.select_dtypes(include=[np.number]).columns.tolist()
numeric_comp  = comp_raw.select_dtypes(include=[np.number]).columns.tolist()

# --- Liv-ex 100 and 1000 from livex parquet ---
lx100_cands  = [c for c in numeric_livex if '100' in str(c) and '1000' not in str(c)]
lx1000_cands = [c for c in numeric_livex if '1000' in str(c)]

LX100_COL  = lx100_cands[0]  if lx100_cands  else (numeric_livex[0] if numeric_livex else None)
LX1000_COL = lx1000_cands[0] if lx1000_cands else None

# --- S&P 500 and FTSE 100 from comparison parquet ---
def find_comp_col(patterns):
    for p in patterns:
        hits = [c for c in numeric_comp if p.lower() in str(c).lower()]
        if hits:
            return hits[0]
    return None

SP500_COL  = find_comp_col(['sp500', 'gspc', 's&p'])
FTSE100_COL = find_comp_col(['ftse100', 'ftse'])

print('=== Column mapping ===')
print(f'  Liv-ex 100:  {LX100_COL}')
print(f'  Liv-ex 1000: {LX1000_COL}')
print(f'  S&P 500:     {SP500_COL}')
print(f'  FTSE 100:    {FTSE100_COL}')

if LX100_COL is None:
    raise ValueError(f'Cannot detect Liv-ex 100 column. Available: {numeric_livex}')
if SP500_COL is None:
    raise ValueError(f'Cannot detect S&P 500 column. Available: {numeric_comp}')

=== Column mapping ===
  Liv-ex 100:  Italy 100
  Liv-ex 1000: Liv-ex Fine Wine 1000
  S&P 500:     sp500
  FTSE 100:    ftse100


## 4. S&P 500 (GBP) — FX Conversion

The comparison parquet holds S&P 500 in USD. Converting to GBP puts it on the same
currency basis as all Liv-ex indices and the custom index.

**Convention**: `GBPUSD=X` gives USD per GBP → `S&P 500 (GBP) = S&P 500 (USD) / GBPUSD`

In [4]:
gbpusd_raw = yf.download('GBPUSD=X', start='2000-01-01', progress=False, auto_adjust=False)['Close']
if isinstance(gbpusd_raw, pd.DataFrame):
    gbpusd_raw = gbpusd_raw.squeeze()
gbpusd_monthly = gbpusd_raw.resample('ME').last()
gbpusd_monthly.name = 'gbpusd'

# Build merged price table
prices = pd.DataFrame(index=comp_raw.index)
prices['S&P 500 (USD)'] = comp_raw[SP500_COL]
if FTSE100_COL:
    prices['FTSE 100'] = comp_raw[FTSE100_COL]

# FX-adjusted S&P 500
merged = prices[['S&P 500 (USD)']].join(gbpusd_monthly, how='left')
merged['gbpusd'] = merged['gbpusd'].ffill(limit=3)
prices['S&P 500 (GBP)'] = merged['S&P 500 (USD)'] / merged['gbpusd']

# Attach Liv-ex indices
for col, label in [(LX100_COL, 'Liv-ex 100'), (LX1000_COL, 'Liv-ex 1000')]:
    if col:
        prices[label] = livex_raw[col].reindex(prices.index, method='ffill')

print(f'Price levels dataset: {prices.shape}')
print(f'Date range: {prices.index.min().date()} → {prices.index.max().date()}')
print('\nNon-null counts:')
print(prices.notna().sum().to_string())
prices.tail(3)

Price levels dataset: (315, 5)
Date range: 2000-01-31 → 2026-03-31

Non-null counts:
S&P 500 (USD)    315
FTSE 100         315
S&P 500 (GBP)    268
Liv-ex 100       268
Liv-ex 1000      268


,S&P 500 (USD),FTSE 100,S&P 500 (GBP),Liv-ex 100,Liv-ex 1000
Date,,,,,
2026-01-31,6939.029785,10223.500000,5025.939317,346.81,350.90
2026-02-28,6878.879883,10910.599609,5098.694646,347.45,351.03
2026-03-31,6781.479980,10412.240234,5121.441278,347.45,351.03


## 5. Build Latent Price Index

**Construction steps**:
1. Pivot LWIN7 monthly latent prices to wide format: rows = month-end, columns = LWIN7.
2. **Normalise per wine**: divide each series by its first non-null latent price x 100
   (first observation = 100 for every constituent).
   `price_latent` is MCMC-modelled -- no further smoothing applied.
3. **Composite**: equal-weighted median across all LWIN7s with non-null normalised values
   for each month. (No volume weights -- the latent model already accounts for trade activity.)
4. **Rebase to 100** at the earliest month where >= `MIN_CONTRIBUTORS = 10`
   LWIN7s have non-null normalised values.

In [ ]:
# Rank LWIN7s by number of months with latent price data
lwin7_rank = (
    lwin7_monthly.groupby('lwin7')['latent_price']
    .count()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'latent_price': 'months_with_data'})
)
lwin7_rank.index += 1  # 1-based rank

all_lwin7s   = lwin7_rank['lwin7'].tolist()
top10_lwin7s = lwin7_rank['lwin7'].head(10).tolist()

print(f'Total LWIN7s in latent price data: {len(all_lwin7s)}')
print(f'Top-10 LWIN7s (by months of latent price coverage):')
display(lwin7_rank.head(10))

In [ ]:
# Pivot to wide format: rows = month, columns = LWIN7
price_wide = lwin7_monthly.pivot_table(
    index='month', columns='lwin7', values='latent_price', aggfunc='first'
)

price_wide.index = pd.to_datetime(price_wide.index)
price_wide.columns.name = None

# Reindex to month-end frequency
full_idx   = pd.date_range(price_wide.index.min(), price_wide.index.max(), freq='ME')
price_wide = price_wide.reindex(full_idx)

print(f'Latent price grid: {price_wide.shape}  (months x LWIN7s)')
print(f'Date range: {price_wide.index.min().date()} -> {price_wide.index.max().date()}')
coverage = price_wide.notna().mean().sort_values(ascending=False)
print(f'Coverage (non-null fraction per LWIN7): mean={coverage.mean():.2f}, min={coverage.min():.2f}')

In [ ]:
# Normalise each LWIN7: first non-null latent price = 100
# price_latent is MCMC-modelled -- no further smoothing applied
norm_wide = pd.DataFrame(index=price_wide.index)
for lwin7 in price_wide.columns:
    series = price_wide[lwin7].dropna()
    if len(series) == 0:
        continue
    base = float(series.iloc[0])
    if base == 0:
        continue
    norm_wide[lwin7] = price_wide[lwin7] / base * 100

# Forward-fill to bridge short calendar gaps (latent model is dense, limit=1)
norm_filled = norm_wide.ffill(limit=1)

print(f'Normalised constituents: {norm_filled.shape}')
print(f'  Median coverage after ffill: {norm_filled.notna().mean().median():.2f}')

In [ ]:
# Equal-weighted composite: median normalised latent price across LWIN7s per month
# (No volume weights -- MCMC model already integrates trade activity)
contributor_count = norm_filled.notna().sum(axis=1)
composite_raw     = norm_filled.median(axis=1, skipna=True)

# Rebase to 100 at earliest month with >= MIN_CONTRIBUTORS LWIN7s
valid_months = contributor_count[contributor_count >= MIN_CONTRIBUTORS]
if len(valid_months) == 0:
    raise ValueError(
        f'No month has >= {MIN_CONTRIBUTORS} contributors. '
        f'Max contributors per month: {int(contributor_count.max())}'
    )

COMPOSITE_REBASE_MONTH = valid_months.index[0]
base_val = float(composite_raw.loc[COMPOSITE_REBASE_MONTH])
composite_index = composite_raw / base_val * 100

# Suppress months with fewer than 5 LWIN7s contributing (unreliable)
composite_index = composite_index.where(contributor_count >= 5)

print(f'Composite rebase month:    {COMPOSITE_REBASE_MONTH.date()}')
print(f'Valid index months:        {composite_index.notna().sum()}')
print(f'Contributor count -- mean: {contributor_count.mean():.1f}, min: {contributor_count.min()}, max: {contributor_count.max()}')
print(f'Date range:                {composite_index.dropna().index.min().date()} -> {composite_index.dropna().index.max().date()}')
composite_index.tail(3)

## 6. Methodology Documentation

### Latent trade price model (MCMC/Kalman)

This notebook constructs a **latent price index** from WineFi's MCMC/Kalman-filtered
trade price model. The model (`dev_winefi_raf.ml.ml_latent_prices_historic`) produces
a `price_latent` estimate per LWIN11 (wine x vintage) per month, derived from:

- **MCMC sampling** of latent price states across observed trade prices
- **Kalman filter** smoothing to propagate price information across months with sparse trading

LWIN11-level latent prices are aggregated to LWIN7 (wine label) by taking the **median
across vintages** for each month. This index is then normalised (each LWIN7 = 100 at
first observation) and combined as an equal-weighted median composite.

---

### Advantages over the raw-trade VWAP approach

| Dimension | Raw-trade VWAP | Latent price model (MCMC/Kalman) |
|-----------|---------------|----------------------------------|
| Smoothness | High noise (sparse months) | MCMC-smoothed, dense coverage |
| Gap handling | Forward-fill heuristic | Kalman state propagation |
| Vintage mixing | VWAP sensitive to composition shifts | Median across vintages per month |
| Model basis | Arithmetic average of transaction prices | Posterior mean of latent price state |

---

### Remaining caveats

> **Honest framing**: The latent price model addresses VWAP noise and gap-filling
> limitations, but the LWIN7 universe still reflects wines with sufficient trade
> activity for model estimation. Wines with very few trades will have wider posterior
> uncertainty in `price_latent`. Directional comparisons with Liv-ex indices are
> valid; absolute level differences require caution.

## 7. Chart 1 — Top-10 Constituent Series

Normalised VWAP for the top-10 LWIN7s by trade count, with the composite index
overlaid in bold. Each series rebased to 100 at its first trade.

In [ ]:
colour_cycle = list(COLOURS.values())

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(
    'Custom Index — Top-10 Normalised Constituent Series\n'
    f'({len(all_lwin7s)} total LWIN7s; composite = volume-weighted; each series rebased to 100 at first trade)',
    fontsize=13, fontweight='bold'
)

for i, lwin7 in enumerate(top10_lwin7s):
    if lwin7 not in norm_filled.columns:
        continue
    series = norm_filled[lwin7].dropna()
    ax.plot(
        series.index, series.values,
        color=colour_cycle[i % len(colour_cycle)],
        linewidth=0.9, alpha=0.5, label=str(lwin7)
    )

# Composite overlay
comp_plot = composite_index.dropna()
ax.plot(
    comp_plot.index, comp_plot.values,
    color='black', linewidth=2.5, zorder=5, label='Composite (volume-weighted)'
)

# Stress period shading
for sp in STRESS_PERIODS:
    ax.axvspan(pd.Timestamp(sp['start']), pd.Timestamp(sp['end']),
               alpha=0.10, color=sp['colour'], zorder=0)

ax.axhline(100, color=COLOURS['slate'], linewidth=0.7, linestyle=':', alpha=0.5)
ax.set_ylabel('Normalised price (100 = first trade)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)

stress_patches = [mpatches.Patch(color=sp['colour'], alpha=0.5, label=sp['label'])
                  for sp in STRESS_PERIODS]
ax.legend(
    handles=[plt.Line2D([0], [0], color='black', linewidth=2.5, label='Composite (equal-weighted median)')] + stress_patches,
    fontsize=9, loc='upper left'
)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
out1 = IMAGES_DIR / '01_constituent_series.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
print(f'Saved → {out1}')
plt.show()

## 8. Chart 2 — Custom Index vs Liv-ex 100 and 1000 (rebased Jan 2005)

All three series rebased to 100 at January 2005 (first available month-end on or after
2005-01-01). Stress periods shaded.

In [ ]:
def rebase_at(series, rebase_date):
    """Rebase series to 100 at the first non-null value on or after rebase_date."""
    s = series.dropna()
    slice_ = s[s.index >= rebase_date]
    if slice_.empty:
        return series
    base = float(slice_.iloc[0])
    if base == 0:
        return series
    return series / base * 100


# Build comparison dataset from REBASE_DATE onwards
comp2 = pd.DataFrame(index=prices.index)
comp2['Latent Price Index'] = composite_index
comp2['Liv-ex 100']   = prices.get('Liv-ex 100')
comp2['Liv-ex 1000']  = prices.get('Liv-ex 1000')
comp2 = comp2[comp2.index >= REBASE_DATE]

# Rebase each column to 100 at REBASE_DATE
for col in comp2.columns:
    comp2[col] = rebase_at(comp2[col], REBASE_DATE)

col_styles = {
    'Latent Price Index': (COLOURS['teal'],   2.5, '-'),
    'Liv-ex 100':   (COLOURS['purple'], 1.8, '-'),
    'Liv-ex 1000':  (COLOURS['blue'],   1.5, '--'),
}

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(
    f'Latent Trade Price Index vs Liv-ex Benchmarks — Rebased Jan {REBASE_DATE[:4]}\n'
    f'MCMC/Kalman latent price composite ({len(all_lwin7s)} LWIN7s) vs Liv-ex 100 & 1000',
    fontsize=13, fontweight='bold'
)

for col, (colour, lw, ls) in col_styles.items():
    if col not in comp2.columns:
        continue
    s = comp2[col].dropna()
    ax.plot(s.index, s.values, color=colour, linewidth=lw, linestyle=ls, label=col, zorder=4)

for sp in STRESS_PERIODS:
    ax.axvspan(pd.Timestamp(sp['start']), pd.Timestamp(sp['end']),
               alpha=0.10, color=sp['colour'], zorder=0)

ax.axhline(100, color=COLOURS['slate'], linewidth=0.7, linestyle=':', alpha=0.5)
ax.set_ylabel(f'Price Index (100 = Jan {REBASE_DATE[:4]})', fontsize=11)
ax.set_xlabel('Date', fontsize=11)

stress_patches = [mpatches.Patch(color=sp['colour'], alpha=0.5, label=sp['label'])
                  for sp in STRESS_PERIODS]
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + stress_patches, fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
out2 = IMAGES_DIR / '02_custom_vs_livex.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f'Saved → {out2}')
plt.show()

## 9. Chart 3 — Rolling 12-Month Return Spread

Rolling 12-month return (%) for the custom index minus Liv-ex 100. Positive months
(custom outperforms) are filled teal; negative (Liv-ex 100 outperforms) are filled
purple.

In [ ]:
if 'Liv-ex 100' not in comp2.columns or comp2['Liv-ex 100'].isna().all():
    print('Liv-ex 100 data unavailable — skipping spread chart.')
else:
    # Rolling 12m return = price[t] / price[t-12] - 1
    custom_r12 = comp2['Latent Price Index'].pct_change(12) * 100
    lx100_r12  = comp2['Liv-ex 100'].pct_change(12) * 100
    spread     = custom_r12 - lx100_r12

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(
        'Custom Index vs Liv-ex 100 — Rolling 12-Month Returns & Spread\n'
        'Positive spread (teal) = latent index outperforms; negative (purple) = Liv-ex 100 outperforms',
        fontsize=13, fontweight='bold'
    )

    # Panel 1: individual rolling returns
    ax1.plot(custom_r12.index, custom_r12.values,
             color=COLOURS['teal'],   linewidth=1.8, label='Latent Price Index (12m return %)')
    ax1.plot(lx100_r12.index, lx100_r12.values,
             color=COLOURS['purple'], linewidth=1.8, label='Liv-ex 100 (12m return %)')
    ax1.axhline(0, color=COLOURS['slate'], linewidth=0.7, linestyle=':')
    ax1.set_ylabel('Rolling 12m return (%)', fontsize=10)
    ax1.legend(fontsize=9, loc='upper left')

    # Panel 2: spread
    ax2.fill_between(spread.index, spread.values, 0,
                     where=(spread.values > 0), alpha=0.40, color=COLOURS['teal'],
                     label='Latent index outperforms')
    ax2.fill_between(spread.index, spread.values, 0,
                     where=(spread.values < 0), alpha=0.40, color=COLOURS['purple'],
                     label='Liv-ex 100 outperforms')
    ax2.plot(spread.index, spread.values,
             color=COLOURS['slate'], linewidth=0.9, alpha=0.6)
    ax2.axhline(0, color=COLOURS['slate'], linewidth=0.8, linestyle=':')
    ax2.set_ylabel('Spread (custom minus Liv-ex 100, pp)', fontsize=10)
    ax2.set_xlabel('Date', fontsize=11)
    ax2.legend(fontsize=9, loc='upper left')

    for ax in [ax1, ax2]:
        for sp in STRESS_PERIODS:
            ax.axvspan(pd.Timestamp(sp['start']), pd.Timestamp(sp['end']),
                       alpha=0.08, color=sp['colour'], zorder=0)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    print(f'\nSpread summary (latent price index minus Liv-ex 100, rolling 12m, pp):')
    print(f'  Mean:   {spread.mean():+.2f}')
    print(f'  Std:    {spread.std():.2f}')
    print(f'  Min:    {spread.min():+.2f}')
    print(f'  Max:    {spread.max():+.2f}')

    plt.tight_layout()
    out3 = IMAGES_DIR / '03_spread_vs_livex100.png'
    plt.savefig(out3, dpi=150, bbox_inches='tight')
    print(f'Saved → {out3}')
    plt.show()

## 10. Chart 4 — Crisis Deep-Dive (3 Windows)

Three subplots — one per stress period — each showing the custom index, Liv-ex 100,
and S&P 500 (GBP) rebased to 100 at the window's start date.

In [ ]:
# Build aligned series for crisis analysis
crisis_series = {
    'Latent Price Index':   composite_index.rename('Latent Price Index'),
    'Liv-ex 100':     prices.get('Liv-ex 100', pd.Series(dtype=float, name='Liv-ex 100')),
    'S&P 500 (GBP)':  prices['S&P 500 (GBP)'],
}

series_colours = {
    'Latent Price Index':  COLOURS['teal'],
    'Liv-ex 100':    COLOURS['purple'],
    'S&P 500 (GBP)': COLOURS['blue'],
}

# Extended plot windows (wider than the stress period for context)
PLOT_BUFFERS = {
    'GFC 2008':        ('2006-01-01', '2012-12-31'),
    'COVID 2020':      ('2019-01-01', '2022-06-30'),
    '2022 Rate Rises': ('2020-07-01', '2024-06-30'),
}

n_panels = len(STRESS_PERIODS)
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6), sharey=False)
fig.suptitle(
    'Crisis Deep-Dive — Custom Index vs Liv-ex 100 vs S&P 500 (GBP)\n'
    'Each series rebased to 100 at window start; stress period shaded',
    fontsize=13, fontweight='bold'
)

for ax, sp in zip(axes, STRESS_PERIODS):
    pstart, pend = PLOT_BUFFERS.get(sp['label'], (sp['start'], sp['end']))
    ps = pd.Timestamp(pstart)
    pe = pd.Timestamp(pend)
    cs = pd.Timestamp(sp['start'])
    ce = pd.Timestamp(sp['end'])

    for name, s in crisis_series.items():
        if s is None or (hasattr(s, 'isna') and s.isna().all()):
            continue
        window = s[(s.index >= ps) & (s.index <= pe)].dropna()
        if len(window) < 3:
            continue
        colour = series_colours.get(name, COLOURS['slate'])
        lw = 2.2 if name == 'Latent Price Index' else 1.6
        indexed = window / window.iloc[0] * 100
        ax.plot(indexed.index, indexed.values, color=colour, linewidth=lw, label=name)

    ax.axvspan(cs, ce, alpha=0.14, color=sp['colour'], zorder=0, label=sp['label'])
    ax.axhline(100, color=COLOURS['slate'], linewidth=0.7, linestyle=':', alpha=0.5)
    ax.set_title(sp['label'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Index (window start = 100)', fontsize=9)
    ax.legend(fontsize=8, loc='lower left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
out4 = IMAGES_DIR / '04_crisis_deepdive.png'
plt.savefig(out4, dpi=150, bbox_inches='tight')
print(f'Saved → {out4}')
plt.show()

## 11. Chart 5 — Crisis Bar Comparison

Grouped bar chart: 3 stress periods × 5 assets (Custom Index, Liv-ex 100, Liv-ex 1000,
S&P 500 (GBP), FTSE 100). Each bar shows the total return during that stress period.

In [ ]:
# Build price table for bar chart (5 assets)
bar_series = {
    'Latent Price Index':  composite_index,
    'Liv-ex 100':    prices.get('Liv-ex 100'),
    'Liv-ex 1000':   prices.get('Liv-ex 1000'),
    'S&P 500 (GBP)': prices['S&P 500 (GBP)'],
    'FTSE 100':      prices.get('FTSE 100'),
}

bar_colours = {
    'Latent Price Index':  COLOURS['teal'],
    'Liv-ex 100':    COLOURS['purple'],
    'Liv-ex 1000':   COLOURS['magenta'],
    'S&P 500 (GBP)': COLOURS['blue'],
    'FTSE 100':      COLOURS['orange'],
}

# Compute total return for each asset × stress period
crisis_stats = []
for sp in STRESS_PERIODS:
    row = {'period': sp['label']}
    for name, s in bar_series.items():
        if s is None or (hasattr(s, 'isna') and s.isna().all()):
            row[name] = np.nan
            continue
        window = s[(s.index >= sp['start']) & (s.index <= sp['end'])].dropna()
        if len(window) < 2:
            row[name] = np.nan
            continue
        ret = (float(window.iloc[-1]) - float(window.iloc[0])) / float(window.iloc[0]) * 100
        row[name] = round(ret, 1)
    crisis_stats.append(row)

crisis_df = pd.DataFrame(crisis_stats).set_index('period')
print('=== Total return during each stress period ===')
display(crisis_df)

In [ ]:
asset_names = [k for k in bar_series if k in crisis_df.columns]
n_periods   = len(crisis_df)
n_assets    = len(asset_names)
x           = np.arange(n_periods)
total_width = 0.75
bar_width   = total_width / n_assets

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle(
    'Total Return During Stress Periods\n'
    'Latent Price Index vs Liv-ex 100, Liv-ex 1000, S&P 500 (GBP), FTSE 100',
    fontsize=13, fontweight='bold'
)

for i, name in enumerate(asset_names):
    vals   = crisis_df[name].values.astype(float)
    offset = (i - n_assets / 2 + 0.5) * bar_width
    colour = bar_colours.get(name, COLOURS['slate'])
    bars   = ax.bar(x + offset, vals, width=bar_width,
                    color=colour, label=name, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        if np.isnan(val):
            continue
        va  = 'bottom' if val >= 0 else 'top'
        ypos = val + (0.6 if val >= 0 else -0.6)
        ax.text(
            bar.get_x() + bar.get_width() / 2, ypos,
            f'{val:+.1f}%',
            ha='center', va=va, fontsize=7.5, color=COLOURS['slate']
        )

ax.axhline(0, color=COLOURS['slate'], linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(crisis_df.index, fontsize=10)
ax.set_ylabel('Total return during stress period (%)', fontsize=10)
ax.legend(fontsize=9, loc='best')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:+.0f}%'))

plt.tight_layout()
out5 = IMAGES_DIR / '05_crisis_bar_comparison.png'
plt.savefig(out5, dpi=150, bbox_inches='tight')
print(f'Saved → {out5}')
plt.show()

## 12. Data Quality Assertions

All checks must pass before this notebook is considered complete.

In [ ]:
errors = []

# 1. All 5 PNGs exist with exact names
REQUIRED_IMAGES = [
    '01_constituent_series.png',
    '02_custom_vs_livex.png',
    '03_spread_vs_livex100.png',
    '04_crisis_deepdive.png',
    '05_crisis_bar_comparison.png',
]
for fname in REQUIRED_IMAGES:
    p = IMAGES_DIR / fname
    if not p.exists():
        errors.append(f'Missing image: {fname}')
    elif p.stat().st_size < 1000:
        errors.append(f'Image too small (likely empty): {fname}')

# 2. Composite index has enough valid months
valid_months_count = int(composite_index.notna().sum())
if valid_months_count < 60:
    errors.append(f'Composite index has only {valid_months_count} valid months (need >= 60)')

# 3. Composite index starts near 100 at rebase month
first_val = float(composite_index.dropna().iloc[0])
if not (90 <= first_val <= 110):
    errors.append(f'Composite index first value {first_val:.1f} is not near 100 -- rebase failed')

# 4. At least 10 LWIN7s in the latent price data
if len(all_lwin7s) < 10:
    errors.append(f'Only {len(all_lwin7s)} LWIN7s in latent price data (need >= 10)')

# 5. Liv-ex 100 column detected
if LX100_COL is None:
    errors.append('Liv-ex 100 column not detected in livex parquet')

# 6. Crisis bar chart has data for at least 2 assets across all periods
non_null_cols = crisis_df.notna().any(axis=0)
if non_null_cols.sum() < 2:
    errors.append(f'Crisis bar chart has data for only {non_null_cols.sum()} assets (need >= 2)')

# 7. images/custom_indices/ directory exists
if not IMAGES_DIR.is_dir():
    errors.append(f'Images directory missing: {IMAGES_DIR}')

# 8. Latent price data sourced from MotherDuck (not parquet)
if 'latent_raw' not in dir() or len(latent_raw) == 0:
    errors.append('latent_raw not populated -- MotherDuck query may have failed')

if errors:
    for err in errors:
        print(f'FAIL: {err}')
    raise AssertionError('Data quality checks failed -- see output above')
else:
    print('All data quality assertions PASSED.')
    print(f'  Data source:            dev_winefi_raf.ml.ml_latent_prices_historic')
    print(f'  Composite rebase month: {COMPOSITE_REBASE_MONTH.date()}')
    print(f'  Composite valid months: {valid_months_count}')
    print(f'  LWIN7 count:            {len(all_lwin7s)}')
    print(f'  Liv-ex 100 column:      {LX100_COL}')
    print(f'  Liv-ex 1000 column:     {LX1000_COL}')
    print(f'  Images saved:')
    for fname in REQUIRED_IMAGES:
        size_kb = (IMAGES_DIR / fname).stat().st_size / 1024
        print(f'    {fname}  ({size_kb:.1f} KB)')

## Summary

| Chart | File | Description |
|-------|------|-------------|
| 1 | `01_constituent_series.png` | Top-10 normalised LWIN7 latent price series + equal-weighted composite |
| 2 | `02_custom_vs_livex.png` | Latent price index vs Liv-ex 100 & 1000, rebased Jan 2005, stress periods shaded |
| 3 | `03_spread_vs_livex100.png` | Rolling 12m return spread: latent price index minus Liv-ex 100 |
| 4 | `04_crisis_deepdive.png` | 3 crisis windows: latent price index vs Liv-ex 100 vs S&P 500 (GBP) |
| 5 | `05_crisis_bar_comparison.png` | Grouped bar: 3 stress periods x 5 assets |

### Index methodology
- **Data source**: `dev_winefi_raf.ml.ml_latent_prices_historic` (MotherDuck)
- **Model**: Latent trade price model (MCMC/Kalman)
- **Aggregation**: LWIN11 -> LWIN7 by median `price_latent` across vintages per month
- **Normalisation**: each LWIN7 series divided by first non-null latent price x 100
- **Composite**: equal-weighted median across LWIN7s with non-null normalised values
- **Rebase anchor**: earliest month with >= 10 contributing LWIN7s
- **Stress periods**: GFC 2008, COVID 2020, 2022 Rate Rises

### Advantages over raw-trade VWAP
- MCMC/Kalman smoothing eliminates sparse-month noise
- Kalman state propagation replaces heuristic forward-fill
- Latent price treats each vintage independently before LWIN7 aggregation

---
*Data source*: `dev_winefi_raf.ml.ml_latent_prices_historic` (MotherDuck)  
*Also depends on*: `01_data_setup.ipynb` (Liv-ex and comparison parquet files in `../data/`)  
*Outputs*: `../images/custom_indices/` -- 5 PNG charts